In [ ]:
import sys
sys.path.append('..')

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from src.config import PATHS
from src.viz import set_style, save_fig, annotate_source
from src.stats import chi2_independence_test, ci_proportions_difference, fit_logistic_model

set_style()

In [ ]:
df_h = pd.read_parquet(PATHS['analytical'] / 'df_hogar_analytic.parquet')
df_p = pd.read_parquet(PATHS['analytical'] / 'df_persona_analytic.parquet')

print(f'Hogares analíticos : {df_h.shape[0]:,} registros')
print(f'Personas analíticas: {df_p.shape[0]:,} registros')

## 9.2 Pruebas de Independencia $\chi^2$ de Pearson y Tamaño del Efecto ($V$ de Cramér)

In [ ]:
test_eco_pc = chi2_independence_test(df_h, 'sit_eco_label', 'tiene_computador')
test_alta_eco = chi2_independence_test(df_h, 'alta_dificultad_eco', 'tiene_computador')
test_jefe_pc = chi2_independence_test(df_h, 'jefe_nivel_educ', 'tiene_computador')

# Nivel persona (excluyendo menores de 5 años que no responden módulo educativo)
valid_p = df_p[df_p['nivel_educ_agrupado'] != 'Menor de 5 años / No aplica']
test_pedu_net = chi2_independence_test(valid_p, 'nivel_educ_agrupado', 'usa_internet')

### Tabla Consolidada de Pruebas $\chi^2$ y V de Cramér

In [ ]:
res_tests = pd.DataFrame([
 {
 'Relación': 'Dificultad Económica (5 cat.) × Computador',
 'N': test_eco_pc['n_observaciones'],
 'Chi2': f"{test_eco_pc['chi2']:.2f}",
 'gl': test_eco_pc['grados_libertad'],
 'p-valor': '< 0.0001',
 'V de Cramér': f"{test_eco_pc['cramer_v']:.4f}",
 'Interpretación del Efecto': test_eco_pc['interpretacion_efecto']
 },
 {
 'Relación': 'Alta Dificultad Económica (binaria) × Computador',
 'N': test_alta_eco['n_observaciones'],
 'Chi2': f"{test_alta_eco['chi2']:.2f}",
 'gl': test_alta_eco['grados_libertad'],
 'p-valor': '< 0.0001',
 'V de Cramér': f"{test_alta_eco['cramer_v']:.4f}",
 'Interpretación del Efecto': test_alta_eco['interpretacion_efecto']
 },
 {
 'Relación': 'Educación del Jefe de Hogar × Computador',
 'N': test_jefe_pc['n_observaciones'],
 'Chi2': f"{test_jefe_pc['chi2']:.2f}",
 'gl': test_jefe_pc['grados_libertad'],
 'p-valor': '< 0.0001',
 'V de Cramér': f"{test_jefe_pc['cramer_v']:.4f}",
 'Interpretación del Efecto': test_jefe_pc['interpretacion_efecto']
 },
 {
 'Relación': 'Educación Individual × Uso de Internet',
 'N': test_pedu_net['n_observaciones'],
 'Chi2': f"{test_pedu_net['chi2']:.2f}",
 'gl': test_pedu_net['grados_libertad'],
 'p-valor': '< 0.0001',
 'V de Cramér': f"{test_pedu_net['cramer_v']:.4f}",
 'Interpretación del Efecto': test_pedu_net['interpretacion_efecto']
 }
])
res_tests

## 9.3 Intervalos de Confianza al 95% para Diferencias de Proporciones

In [ ]:
# Brecha de computador por vulnerabilidad económica
n_sin_dif = (df_h['alta_dificultad_eco'] == 0).sum()
x_sin_dif = df_h[df_h['alta_dificultad_eco'] == 0]['tiene_computador'].sum()
n_con_dif = (df_h['alta_dificultad_eco'] == 1).sum()
x_con_dif = df_h[df_h['alta_dificultad_eco'] == 1]['tiene_computador'].sum()
ci_pc_dif = ci_proportions_difference(x_sin_dif, n_sin_dif, x_con_dif, n_con_dif)

# Brecha de uso de internet por educación superior vs sin educación
p_sup = df_p[df_p['nivel_educ_agrupado'] == '5. Superior / Universitario']
p_ning = df_p[df_p['nivel_educ_agrupado'] == '1. Ninguno / Preescolar']
ci_edu_dif = ci_proportions_difference(p_sup['usa_internet'].sum(), len(p_sup), p_ning['usa_internet'].sum(), len(p_ning))

print(f'1. Brecha de Computador (Sin dificultad vs. Alta dificultad):')
print(f' Δp = {ci_pc_dif["diferencia_pp"]:.2f} pp | IC 95%: [{ci_pc_dif["ci_lower_pp"]:.2f} pp, {ci_pc_dif["ci_upper_pp"]:.2f} pp] | Z = {ci_pc_dif["z_stat"]:.2f}')

print(f'\n2. Brecha de Uso de Internet (Educación Superior vs. Ninguno):')
print(f' Δp = {ci_edu_dif["diferencia_pp"]:.2f} pp | IC 95%: [{ci_edu_dif["ci_lower_pp"]:.2f} pp, {ci_edu_dif["ci_upper_pp"]:.2f} pp] | Z = {ci_edu_dif["z_stat"]:.2f}')

## 9.4 Modelo de Regresión Logística Multivariada (Hogar: Tenencia de Computador)

In [ ]:
educ_valid = ['1. Ninguno / Preescolar', '2. Primaria', '3. Secundaria', '4. Media', '5. Superior / Universitario']
df_h_reg = df_h[df_h['jefe_nivel_educ'].isin(educ_valid)].copy()

formula_h = "tiene_computador ~ alta_dificultad_eco + es_urbano + n_personas_hogar + C(jefe_nivel_educ, Treatment(reference='1. Ninguno / Preescolar'))"
mod_h = fit_logistic_model(df_h_reg, formula_h)
mod_h[['Odds Ratio (OR)', 'OR CI 2.5%', 'OR CI 97.5%', 'p-valor']]

### Gráfico 10: Forest Plot — Determinantes de la Tenencia de Computador en el Hogar

In [ ]:
labels_map = {
 "C(jefe_nivel_educ, Treatment(reference='1. Ninguno / Preescolar'))[T.2. Primaria]": 'Jefe: Primaria (vs. Ninguno)',
 "C(jefe_nivel_educ, Treatment(reference='1. Ninguno / Preescolar'))[T.3. Secundaria]": 'Jefe: Secundaria (vs. Ninguno)',
 "C(jefe_nivel_educ, Treatment(reference='1. Ninguno / Preescolar'))[T.4. Media]": 'Jefe: Media (vs. Ninguno)',
 "C(jefe_nivel_educ, Treatment(reference='1. Ninguno / Preescolar'))[T.5. Superior / Universitario]": 'Jefe: Superior / Univ. (vs. Ninguno)',
 'alta_dificultad_eco': 'Alta dificultad económica del hogar',
 'es_urbano': 'Dominio Urbano (Cabecera vs. Rural)',
 'n_personas_hogar': 'Tamaño del hogar (+1 persona)'
}

plot_df = mod_h.loc[list(labels_map.keys())].copy()
plot_df.index = [labels_map[k] for k in plot_df.index]
plot_df = plot_df.iloc[::-1]

fig, ax = plt.subplots(figsize=(9, 5.2))
y_pos = np.arange(len(plot_df))
ax.axvline(1.0, color='gray', linestyle='--', linewidth=1, alpha=0.7)

ax.errorbar(
 plot_df['Odds Ratio (OR)'], y_pos,
 xerr=[plot_df['Odds Ratio (OR)'] - plot_df['OR CI 2.5%'], plot_df['OR CI 97.5%'] - plot_df['Odds Ratio (OR)']],
 fmt='o', color='#1f77b4', ecolor='#2b5c8f', elinewidth=2, capsize=4, markersize=7
)

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df.index, fontsize=9.5)
ax.set_xlabel('Odds Ratio (OR) con Intervalo de Confianza al 95% (Escala Logarítmica)', labelpad=10)
ax.set_xscale('log')
ax.set_title('Determinantes de la Tenencia de Computador en el Hogar (Modelo Logístico)', pad=15)

for i, (idx, row) in enumerate(plot_df.iterrows()):
 ax.text(row['OR CI 97.5%'] * 1.08, i, f"{row['Odds Ratio (OR)']:.2f} [{row['OR CI 2.5%']:.2f}, {row['OR CI 97.5%']:.2f}]", 
 va='center', fontsize=8.5, fontweight='bold', color='#1f77b4')

plt.tight_layout()
annotate_source(ax, 'Fuente: DANE — ECV 2024. Regresión logística multivariada. N = 83,559 hogares.')
save_fig(fig, 'fig10_odds_ratios_computador_hogar')
plt.show()

## 9.5 Modelo de Regresión Logística Multivariada (Persona: Uso Individual de Internet)

In [ ]:
df_p_reg = df_p[df_p['nivel_educ_agrupado'].isin(educ_valid)].copy()
formula_p = "usa_internet ~ alta_dificultad_eco + es_urbano + tiene_celular + C(nivel_educ_agrupado, Treatment(reference='1. Ninguno / Preescolar'))"
mod_p = fit_logistic_model(df_p_reg, formula_p)
mod_p[['Odds Ratio (OR)', 'OR CI 2.5%', 'OR CI 97.5%', 'p-valor']]

### Gráfico 11: Forest Plot — Determinantes del Uso Individual de Internet

In [ ]:
labels_map_p = {
 "C(nivel_educ_agrupado, Treatment(reference='1. Ninguno / Preescolar'))[T.2. Primaria]": 'Educación: Primaria (vs. Ninguno)',
 "C(nivel_educ_agrupado, Treatment(reference='1. Ninguno / Preescolar'))[T.3. Secundaria]": 'Educación: Secundaria (vs. Ninguno)',
 "C(nivel_educ_agrupado, Treatment(reference='1. Ninguno / Preescolar'))[T.4. Media]": 'Educación: Media (vs. Ninguno)',
 "C(nivel_educ_agrupado, Treatment(reference='1. Ninguno / Preescolar'))[T.5. Superior / Universitario]": 'Educación: Superior (vs. Ninguno)',
 'alta_dificultad_eco': 'Alta dificultad económica del hogar',
 'es_urbano': 'Dominio Urbano (vs. Rural)',
 'tiene_celular': 'Posee celular propio (vs. No tiene)'
}

plot_p = mod_p.loc[list(labels_map_p.keys())].copy()
plot_p.index = [labels_map_p[k] for k in plot_p.index]
plot_p = plot_p.iloc[::-1]

fig, ax = plt.subplots(figsize=(9, 5.2))
y_pos = np.arange(len(plot_p))
ax.axvline(1.0, color='gray', linestyle='--', linewidth=1, alpha=0.7)

ax.errorbar(
 plot_p['Odds Ratio (OR)'], y_pos,
 xerr=[plot_p['Odds Ratio (OR)'] - plot_p['OR CI 2.5%'], plot_p['OR CI 97.5%'] - plot_p['Odds Ratio (OR)']],
 fmt='s', color='#2ca02c', ecolor='#006d2c', elinewidth=2, capsize=4, markersize=7
)

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_p.index, fontsize=9.5)
ax.set_xlabel('Odds Ratio (OR) con Intervalo de Confianza al 95% (Escala Logarítmica)', labelpad=10)
ax.set_xscale('log')
ax.set_title('Determinantes del Uso Individual de Internet (Modelo Logístico)', pad=15)

for i, (idx, row) in enumerate(plot_p.iterrows()):
 ax.text(row['OR CI 97.5%'] * 1.08, i, f"{row['Odds Ratio (OR)']:.2f} [{row['OR CI 2.5%']:.2f}, {row['OR CI 97.5%']:.2f}]", 
 va='center', fontsize=8.5, fontweight='bold', color='#006d2c')

plt.tight_layout()
annotate_source(ax, 'Fuente: DANE — ECV 2024. Regresión logística multivariada. N = 162,112 personas.')
save_fig(fig, 'fig11_odds_ratios_uso_internet_persona')
plt.show()